# 05 — Advanced Topics

The previous notebooks built a complete engagement model: detect, compute intercept, simulate pursuit, analyze geometry limits. This notebook extends that foundation in three directions that show up immediately when the problem gets harder.

**Maneuvering targets** — what happens when the target changes heading mid-flight? The constant-velocity intercept is now wrong the moment the target turns. How do you recover?

**Proportional navigation** — the guidance law used in real intercept systems. Instead of re-aiming at the target's current position (pure pursuit) or a predicted future position (constant-velocity), proportional navigation steers to null out the rate of change of bearing. It is provably more efficient than pure pursuit for crossing geometries.

**Multi-threat prioritization** — when multiple targets are inbound simultaneously, which do you engage first? Time to impact, angular rate, distance — different metrics give different answers.

Each section can be read independently. They share the same helpers from earlier notebooks.

In [ ]:
import math
import matplotlib.pyplot as plt
from ipyleaflet import Map, GeoJSON

# ── Geometry helpers ──────────────────────────────────────────────────────────

def compute_bearing(p1, p2):
    lon1, lat1 = math.radians(p1[0]), math.radians(p1[1])
    lon2, lat2 = math.radians(p2[0]), math.radians(p2[1])
    d_lon = lon2 - lon1
    x = math.sin(d_lon) * math.cos(lat2)
    y = math.cos(lat1) * math.sin(lat2) - math.sin(lat1) * math.cos(lat2) * math.cos(d_lon)
    return (math.degrees(math.atan2(x, y)) + 360) % 360

def haversine_km(p1, p2):
    R = 6371.0
    lon1, lat1 = math.radians(p1[0]), math.radians(p1[1])
    lon2, lat2 = math.radians(p2[0]), math.radians(p2[1])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = math.sin(dlat/2)**2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon/2)**2
    return R * 2 * math.asin(math.sqrt(a))

def destination_point(origin, bearing_deg, distance_km):
    R   = 6371.0
    d   = distance_km / R
    brg = math.radians(bearing_deg)
    lat1 = math.radians(origin[1])
    lon1 = math.radians(origin[0])
    lat2 = math.asin(math.sin(lat1)*math.cos(d) + math.cos(lat1)*math.sin(d)*math.cos(brg))
    lon2 = lon1 + math.atan2(math.sin(brg)*math.sin(d)*math.cos(lat1),
                              math.cos(d) - math.sin(lat1)*math.sin(lat2))
    return [math.degrees(lon2), math.degrees(lat2)]

def find_intercept_time(s_pos, t_pos, t_hdg, t_spd, s_spd, t_max=10.0, tol=1e-6):
    def f(t):
        return haversine_km(s_pos, destination_point(t_pos, t_hdg, t_spd*t)) - s_spd*t
    if f(t_max) > 0: return None
    lo, hi = 0.0, t_max
    for _ in range(60):
        mid = (lo + hi) / 2
        if f(mid) > 0: lo = mid
        else: hi = mid
        if hi - lo < tol: break
    return (lo + hi) / 2

def path_length_km(path):
    return sum(haversine_km(path[i], path[i+1]) for i in range(len(path)-1))

def fc(features):
    return {"type": "FeatureCollection", "features": features}

def geoline(coords, props={}):
    return {"type":"Feature","geometry":{"type":"LineString","coordinates":coords},"properties":props}

def geopoint(coord, props={}):
    return {"type":"Feature","geometry":{"type":"Point","coordinates":coord},"properties":props}

print("Helpers loaded.")

## 1. Maneuvering Targets

The constant-velocity intercept assumes the target travels in a straight line. The moment the target turns, the computed intercept point is wrong. Pure pursuit handles this automatically — it re-aims at every step — but at the cost of a curved path and extra distance.

A more realistic model: the **re-intercept loop**. At each guidance update interval, re-solve the intercept from the current pursuer position to the current target position. This gives the efficiency of straight-line flight with the adaptability of pursuit.

The key parameter is the **update interval**: how frequently the pursuer recomputes. More frequent updates → more accurate tracking of a maneuvering target, but also more course corrections.

In [ ]:
def simulate_maneuvering_target(s_pos, t_pos, maneuver_schedule, t_spd, s_spd,
                                update_interval=0.05, dt=0.005,
                                max_steps=6000, capture_km=5.0):
    """
    Simulate pursuit against a target that changes heading at scheduled times.

    maneuver_schedule: list of (time_hours, new_heading) tuples, sorted by time.
    update_interval:   how often (hours) the pursuer re-solves the intercept.
    dt:                simulation time step (hours).

    Returns dict with pursuer_path, target_path, captured, time_elapsed.
    """
    pursuer  = list(s_pos)
    target   = list(t_pos)
    p_path   = [list(pursuer)]
    t_path   = [list(target)]

    # Current target heading — starts with the first scheduled heading
    t_hdg = maneuver_schedule[0][1]
    schedule = list(maneuver_schedule)   # mutable copy

    # Initial intercept bearing
    t_int = find_intercept_time(pursuer, target, t_hdg, t_spd, s_spd, t_max=10.0)
    if t_int is None:
        current_bearing = compute_bearing(pursuer, target)   # fallback: pure pursuit
    else:
        ipt = destination_point(target, t_hdg, t_spd * t_int)
        current_bearing = compute_bearing(pursuer, ipt)

    time_since_update = 0.0
    elapsed = 0.0

    for step in range(max_steps):
        dist = haversine_km(pursuer, target)
        if dist <= capture_km:
            return {"pursuer_path": p_path, "target_path": t_path,
                    "captured": True, "time_elapsed": elapsed}

        # Apply any scheduled heading changes
        while schedule and elapsed >= schedule[0][0]:
            t_hdg = schedule.pop(0)[1]

        # Re-solve intercept at update intervals
        if time_since_update >= update_interval:
            t_int = find_intercept_time(pursuer, target, t_hdg, t_spd, s_spd, t_max=10.0)
            if t_int is not None:
                ipt = destination_point(target, t_hdg, t_spd * t_int)
                current_bearing = compute_bearing(pursuer, ipt)
            else:
                current_bearing = compute_bearing(pursuer, target)   # fallback
            time_since_update = 0.0

        pursuer = destination_point(pursuer, current_bearing, s_spd * dt)
        target  = destination_point(target,  t_hdg,           t_spd * dt)
        p_path.append(list(pursuer))
        t_path.append(list(target))
        elapsed          += dt
        time_since_update += dt

    return {"pursuer_path": p_path, "target_path": t_path,
            "captured": False, "time_elapsed": elapsed}


# ── Scenario: target makes a sharp turn partway through ──────────────────────

s_pos = [-98.49, 33.91]   # Wichita Falls
t_pos = [-101.0, 36.5]
t_spd = 300
s_spd = 650

# Target flies SE for 0.1 h, then turns sharply NE
maneuver_schedule = [
    (0.0,  135),   # heading SE at start
    (0.1,   45),   # turn NE at t = 6 min
]

result_maneuvering = simulate_maneuvering_target(
    s_pos, t_pos, maneuver_schedule, t_spd, s_spd,
    update_interval=0.02, dt=0.005
)

cap  = result_maneuvering["captured"]
tof  = result_maneuvering["time_elapsed"] * 60
dist = path_length_km(result_maneuvering["pursuer_path"])
print(f"Maneuvering target:  captured={cap}  TOF={tof:.1f} min  pursuer path={dist:.1f} km")

In [ ]:
# Compare: straight-line intercept (no re-solve) vs re-intercept loop vs pure pursuit

def simulate_pure_pursuit(s_pos, t_pos, maneuver_schedule, t_spd, s_spd,
                           dt=0.005, max_steps=6000, capture_km=5.0):
    """Pure pursuit against a maneuvering target."""
    pursuer = list(s_pos)
    target  = list(t_pos)
    p_path  = [list(pursuer)]
    t_path  = [list(target)]
    schedule = list(maneuver_schedule)
    t_hdg = schedule[0][1]
    elapsed = 0.0
    for step in range(max_steps):
        if haversine_km(pursuer, target) <= capture_km:
            return {"pursuer_path": p_path, "target_path": t_path,
                    "captured": True, "time_elapsed": elapsed}
        while schedule and elapsed >= schedule[0][0]:
            t_hdg = schedule.pop(0)[1]
        brg = compute_bearing(pursuer, target)
        pursuer = destination_point(pursuer, brg,   s_spd * dt)
        target  = destination_point(target,  t_hdg, t_spd * dt)
        p_path.append(list(pursuer))
        t_path.append(list(target))
        elapsed += dt
    return {"pursuer_path": p_path, "target_path": t_path,
            "captured": False, "time_elapsed": elapsed}


# Re-intercept with fast updates
result_fast = simulate_maneuvering_target(
    s_pos, t_pos, maneuver_schedule, t_spd, s_spd,
    update_interval=0.005, dt=0.005
)

# Re-intercept with slow updates (12-min intervals)
result_slow = simulate_maneuvering_target(
    s_pos, t_pos, maneuver_schedule, t_spd, s_spd,
    update_interval=0.2, dt=0.005
)

# Pure pursuit
result_pursuit = simulate_pure_pursuit(
    s_pos, t_pos, maneuver_schedule, t_spd, s_spd, dt=0.005
)

rows = [
    ("Re-intercept (fast, ~18s)",  result_fast),
    ("Re-intercept (slow, 12min)", result_slow),
    ("Pure pursuit",               result_pursuit),
]

print(f"{'Method':<32}  {'Captured':>9}  {'TOF (min)':>10}  {'Path (km)':>10}")
print("-" * 68)
for label, r in rows:
    cap  = "yes" if r["captured"] else "NO"
    tof  = f"{r['time_elapsed']*60:.1f}" if r["captured"] else "—"
    dist = f"{path_length_km(r['pursuer_path']):.1f}" if r["captured"] else "—"
    print(f"{label:<32}  {cap:>9}  {tof:>10}  {dist:>10}")

In [ ]:
# Map all three paths

maneuver_map = Map(center=(35.5, -99.5), zoom=6)

colors = ["#2a9d8f", "#f4a261", "#e63946"]
labels = ["Re-intercept fast", "Re-intercept slow", "Pure pursuit"]
results_list = [result_fast, result_slow, result_pursuit]

for result, color, label in zip(results_list, colors, labels):
    step = max(1, len(result["pursuer_path"]) // 300)
    maneuver_map.add(GeoJSON(
        data=fc([geoline(result["pursuer_path"][::step])]),
        style={"color": color, "weight": 2}
    ))

# Target path (same for all — use fast result)
step = max(1, len(result_fast["target_path"]) // 300)
maneuver_map.add(GeoJSON(
    data=fc([geoline(result_fast["target_path"][::step])]),
    style={"color": "#457b9d", "weight": 1.5, "dashArray": "5"}
))

# Mark turn point
turn_time_hr = maneuver_schedule[1][0]
turn_dist_km = t_spd * turn_time_hr
turn_pos = destination_point(t_pos, maneuver_schedule[0][1], turn_dist_km)
maneuver_map.add(GeoJSON(
    data=fc([geopoint(turn_pos)]),
    point_style={"radius": 7, "color": "#e9c46a", "fillOpacity": 1.0}
))

# Start points
maneuver_map.add(GeoJSON(
    data=fc([geopoint(s_pos)]),
    point_style={"radius": 8, "color": "#000", "fillOpacity": 1.0}
))
maneuver_map.add(GeoJSON(
    data=fc([geopoint(t_pos)]),
    point_style={"radius": 8, "color": "#457b9d", "fillOpacity": 1.0}
))

print("Teal = re-intercept fast | Orange = re-intercept slow | Red = pure pursuit")
print("Blue dashes = target path | Yellow = target turn point")
maneuver_map

## 2. Proportional Navigation

Pure pursuit always points at where the target *is*. Proportional navigation (PN) points toward where the target *is going* by steering to keep the **line-of-sight (LOS) bearing constant**.

The idea: if the bearing from pursuer to target is not changing, a collision is guaranteed. If it is rotating, steer to cancel the rotation. The steering command is proportional to the LOS rate:

```text
heading_rate = N × LOS_rate
```

where `N` is the navigation constant (typically 3–5). A higher N reacts faster but can overshoot.

Implemented discretely: at each step, compute the change in bearing from pursuer to target since the last step. Rotate the pursuer heading by `N × Δbearing`.

In [ ]:
def bearing_diff(b1, b2):
    """Signed shortest angular difference b2 - b1, in [-180, 180)."""
    d = (b2 - b1 + 360) % 360
    return d - 360 if d >= 180 else d


def simulate_proportional_nav(s_pos, t_pos, t_hdg, t_spd, s_spd,
                               N=4, dt=0.005, max_steps=4000, capture_km=5.0):
    """
    Proportional navigation guidance.

    The pursuer's heading rotates at N times the LOS rate each step.
    Returns same dict shape as other simulators.
    """
    pursuer = list(s_pos)
    target  = list(t_pos)
    p_path  = [list(pursuer)]
    t_path  = [list(target)]

    # Initialize heading toward the target
    heading = compute_bearing(pursuer, target)
    prev_los = heading

    for step in range(max_steps):
        dist = haversine_km(pursuer, target)
        if dist <= capture_km:
            return {"pursuer_path": p_path, "target_path": t_path,
                    "captured": True, "time_elapsed": step * dt}

        current_los = compute_bearing(pursuer, target)
        los_rate    = bearing_diff(prev_los, current_los)   # deg per dt
        heading     = (heading + N * los_rate) % 360

        pursuer  = destination_point(pursuer, heading, s_spd * dt)
        target   = destination_point(target,  t_hdg,   t_spd * dt)
        p_path.append(list(pursuer))
        t_path.append(list(target))
        prev_los = current_los

    return {"pursuer_path": p_path, "target_path": t_path,
            "captured": False, "time_elapsed": max_steps * dt}


# ── Compare PN vs pure pursuit on a crossing geometry ────────────────────────

def simulate_pure_pursuit_straight(s_pos, t_pos, t_hdg, t_spd, s_spd,
                                    dt=0.005, max_steps=4000, capture_km=5.0):
    pursuer = list(s_pos)
    target  = list(t_pos)
    p_path  = [list(pursuer)]
    t_path  = [list(target)]
    for step in range(max_steps):
        if haversine_km(pursuer, target) <= capture_km:
            return {"pursuer_path": p_path, "target_path": t_path,
                    "captured": True, "time_elapsed": step * dt}
        brg    = compute_bearing(pursuer, target)
        pursuer = destination_point(pursuer, brg,   s_spd * dt)
        target  = destination_point(target,  t_hdg, t_spd * dt)
        p_path.append(list(pursuer))
        t_path.append(list(target))
    return {"pursuer_path": p_path, "target_path": t_path,
            "captured": False, "time_elapsed": max_steps * dt}


s_pos = [-98.49, 33.91]
t_pos = [-96.5,  36.5]   # target NE, crossing from E heading W
t_hdg = 270              # heading due west — crossing path
t_spd = 300
s_spd = 600

result_pn  = simulate_proportional_nav(s_pos, t_pos, t_hdg, t_spd, s_spd, N=4)
result_pp  = simulate_pure_pursuit_straight(s_pos, t_pos, t_hdg, t_spd, s_spd)

# Also compute constant-velocity intercept for reference
t_int = find_intercept_time(s_pos, t_pos, t_hdg, t_spd, s_spd)
ipt = destination_point(t_pos, t_hdg, t_spd * t_int) if t_int else None

print(f"{'Method':<28}  {'Captured':>9}  {'TOF (min)':>10}  {'Path (km)':>10}")
print("-" * 65)
for label, r in [("Proportional nav (N=4)", result_pn), ("Pure pursuit", result_pp)]:
    cap  = "yes" if r["captured"] else "NO"
    tof  = f"{r['time_elapsed']*60:.1f}" if r["captured"] else "—"
    dist = f"{path_length_km(r['pursuer_path']):.1f}" if r["captured"] else "—"
    print(f"{label:<28}  {cap:>9}  {tof:>10}  {dist:>10}")
if t_int:
    print(f"{'Constant-vel intercept':<28}  {'yes':>9}  {t_int*60:>10.1f}  "
          f"{haversine_km(s_pos, ipt):>10.1f}")

In [ ]:
# Map: PN vs pure pursuit on crossing geometry

pn_map = Map(center=(35.2, -98.0), zoom=7)

step_pn = max(1, len(result_pn["pursuer_path"]) // 300)
step_pp = max(1, len(result_pp["pursuer_path"]) // 300)

pn_map.add(GeoJSON(
    data=fc([geoline(result_pn["pursuer_path"][::step_pn])]),
    style={"color": "#2a9d8f", "weight": 2.5, "dashArray": "none"}
))
pn_map.add(GeoJSON(
    data=fc([geoline(result_pp["pursuer_path"][::step_pp])]),
    style={"color": "#e63946", "weight": 2.5}
))
pn_map.add(GeoJSON(
    data=fc([geoline(result_pn["target_path"][::step_pn])]),
    style={"color": "#457b9d", "weight": 1.5, "dashArray": "5"}
))
if ipt:
    pn_map.add(GeoJSON(
        data=fc([geoline([s_pos, ipt])]),
        style={"color": "#555", "weight": 1.5, "dashArray": "8"}
    ))
    pn_map.add(GeoJSON(
        data=fc([geopoint(ipt)]),
        point_style={"radius": 6, "color": "#555", "fillOpacity": 0.9}
    ))

pn_map.add(GeoJSON(
    data=fc([geopoint(s_pos)]),
    point_style={"radius": 8, "color": "#000", "fillOpacity": 1.0}
))
pn_map.add(GeoJSON(
    data=fc([geopoint(t_pos)]),
    point_style={"radius": 8, "color": "#457b9d", "fillOpacity": 1.0}
))

print("Teal = proportional nav | Red = pure pursuit | Blue dashes = target | Gray dashes = CV intercept line")
pn_map

## 3. Multi-Threat Prioritization

When multiple threats are inbound simultaneously, a defender must decide engagement order. Three common criteria:

| Criterion | Logic | Risk |
|-----------|-------|------|
| **Nearest** | Engage the closest target first | Ignores heading — a close target moving away is less urgent than a far one approaching fast |
| **Shortest TOF** | Engage the target with the fastest intercept | Computationally intensive; doesn't account for damage priority |
| **Time to impact** (TTI) | Engage the target that will arrive first if uncontested | Most operationally intuitive for area defense |

This section computes all three rankings and shows where they agree and disagree.

In [ ]:
DEFENDED_POINT = [-98.49, 33.91]   # Wichita Falls — asset being defended
SHOOTER_POS    = [-99.5,  34.8]    # defender launch position (offset from asset)
SHOOTER_SPEED  = 700               # km/h

threats = [
    {"id": "T1", "pos": [-96.0,  36.0], "hdg": 225, "spd": 350,
     "description": "fast, heading SW toward asset"},
    {"id": "T2", "pos": [-102.0, 37.0], "hdg": 135, "spd": 250,
     "description": "moderate, heading SE — will miss slightly"},
    {"id": "T3", "pos": [-98.0,  37.5], "hdg": 180, "spd": 200,
     "description": "slow, heading directly south toward asset"},
    {"id": "T4", "pos": [-95.5,  35.0], "hdg": 270, "spd": 400,
     "description": "fast, heading west — crossing geometry"},
]


def time_to_impact(t_pos, t_hdg, t_spd, defended, max_t=20.0, tol=1e-4):
    """
    Estimate when the threat would reach the defended point if uncontested.
    Uses binary search on the minimum approach distance.
    Returns hours, or None if the threat never comes within 20 km.
    """
    # Sample the range over time and find minimum
    best_dist = float("inf")
    best_t    = None
    for i in range(int(max_t / tol)):
        t = i * tol
        pos = destination_point(t_pos, t_hdg, t_spd * t)
        d   = haversine_km(pos, defended)
        if d < best_dist:
            best_dist = d
            best_t    = t
        elif d > best_dist + 50:   # moving away — past closest approach
            break
    return best_t if best_dist < 30 else None


results = []
for thr in threats:
    dist_now = haversine_km(SHOOTER_POS, thr["pos"])
    tof      = find_intercept_time(SHOOTER_POS, thr["pos"], thr["hdg"],
                                   thr["spd"], SHOOTER_SPEED, t_max=15.0)
    tti      = time_to_impact(thr["pos"], thr["hdg"], thr["spd"], DEFENDED_POINT)
    results.append({
        "id":          thr["id"],
        "dist_km":     dist_now,
        "tof_min":     tof * 60 if tof else None,
        "tti_min":     tti * 60 if tti else None,
        "description": thr["description"],
    })

print(f"{'ID':<5}  {'Dist (km)':>10}  {'Intercept TOF':>14}  {'Time to impact':>15}  Description")
print("-" * 80)
for r in results:
    tof_str = f"{r['tof_min']:.1f} min" if r["tof_min"] else "impossible"
    tti_str = f"{r['tti_min']:.1f} min" if r["tti_min"] else "no impact"
    print(f"{r['id']:<5}  {r['dist_km']:>9.1f} km  {tof_str:>14}  {tti_str:>15}  {r['description']}")

print()
print("— Nearest first:         ", sorted(results, key=lambda x: x["dist_km"])[0]["id"])
valid_tof = [r for r in results if r["tof_min"]]
print("— Shortest intercept TOF:", sorted(valid_tof, key=lambda x: x["tof_min"])[0]["id"] if valid_tof else "none")
valid_tti = [r for r in results if r["tti_min"]]
print("— Earliest impact (TTI): ", sorted(valid_tti, key=lambda x: x["tti_min"])[0]["id"] if valid_tti else "none")

In [ ]:
# Map: threats, intercept lines, projected paths, defended asset

threat_map = Map(center=(35.5, -98.5), zoom=6)

threat_colors = {"T1": "#e63946", "T2": "#f4a261", "T3": "#2a9d8f", "T4": "#9b5de5"}

for thr, res in zip(threats, results):
    color = threat_colors[thr["id"]]

    # Projected threat path (uncontested, 600 km)
    track_end = destination_point(thr["pos"], thr["hdg"], 600)
    threat_map.add(GeoJSON(
        data=fc([geoline([thr["pos"], track_end])]),
        style={"color": color, "weight": 1.5, "dashArray": "5", "opacity": 0.6}
    ))

    # Intercept line
    if res["tof_min"]:
        t_int = res["tof_min"] / 60
        ipt   = destination_point(thr["pos"], thr["hdg"], thr["spd"] * t_int)
        threat_map.add(GeoJSON(
            data=fc([geoline([SHOOTER_POS, ipt])]),
            style={"color": color, "weight": 2}
        ))
        threat_map.add(GeoJSON(
            data=fc([geopoint(ipt)]),
            point_style={"radius": 5, "color": color, "fillOpacity": 1.0}
        ))

    # Threat start marker
    threat_map.add(GeoJSON(
        data=fc([geopoint(thr["pos"])]),
        point_style={"radius": 8, "color": color, "fillOpacity": 0.9}
    ))

# Shooter
threat_map.add(GeoJSON(
    data=fc([geopoint(SHOOTER_POS)]),
    point_style={"radius": 9, "color": "#000", "fillOpacity": 1.0}
))

# Defended asset
threat_map.add(GeoJSON(
    data=fc([geopoint(DEFENDED_POINT)]),
    point_style={"radius": 10, "color": "#e9c46a", "fillOpacity": 1.0}
))

print("Yellow = defended asset | Black = shooter | Colors = T1–T4 (dashes = projected path, solid = intercept)")
threat_map

---

## Exercise A — Navigation Constant Sensitivity

The proportional navigation constant `N` controls how aggressively the pursuer corrects for LOS rate. Test `N` values of `2, 3, 4, 5, 6, 8` on the crossing geometry from Section 2.

For each `N`, record: captured (yes/no), TOF, and path length.

Plot `N` vs. path length. Then answer:

1. What happens at very low N (e.g., 2)? Why might the pursuer fail to converge?
2. What happens at very high N (e.g., 8)? Does more aggressive steering always help?
3. Which value of N gives performance closest to the constant-velocity intercept?

In [ ]:
# Exercise A

s_pos = [-98.49, 33.91]
t_pos = [-96.5,  36.5]
t_hdg = 270
t_spd = 300
s_spd = 600

N_values = [2, 3, 4, 5, 6, 8]

print(f"{'N':>4}  {'Captured':>9}  {'TOF (min)':>10}  {'Path (km)':>10}")
print("-" * 42)

path_lengths = []
for N in N_values:
    r = simulate_proportional_nav(s_pos, t_pos, t_hdg, t_spd, s_spd, N=N)
    cap  = "yes" if r["captured"] else "NO"
    tof  = f"{r['time_elapsed']*60:.1f}" if r["captured"] else "—"
    pl   = path_length_km(r["pursuer_path"]) if r["captured"] else None
    dist = f"{pl:.1f}" if pl else "—"
    path_lengths.append(pl)
    print(f"{N:>4}  {cap:>9}  {tof:>10}  {dist:>10}")

# CV intercept reference
t_cv = find_intercept_time(s_pos, t_pos, t_hdg, t_spd, s_spd)
cv_dist = haversine_km(s_pos, destination_point(t_pos, t_hdg, t_spd * t_cv)) if t_cv else None
if cv_dist:
    print(f"\nCV intercept reference:  {cv_dist:.1f} km  ({t_cv*60:.1f} min)")

# Plot
valid_N    = [n for n, pl in zip(N_values, path_lengths) if pl is not None]
valid_dist = [pl for pl in path_lengths if pl is not None]

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(valid_N, valid_dist, "o-", color="#2a9d8f", markersize=7)
if cv_dist:
    ax.axhline(cv_dist, color="#888", linestyle="--", linewidth=1, label=f"CV intercept ({cv_dist:.0f} km)")
ax.set_xlabel("Navigation constant N")
ax.set_ylabel("Pursuer path length (km)")
ax.set_title("PN path length vs. navigation constant  (crossing geometry)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Exercise B — Evasive Maneuver Timing

A target makes a single turn to try to break the intercept. The question is: *when* during the flight is the turn most effective?

Use the maneuvering target simulator from Section 1. Fix the scenario (WF shooter, NW target, initial heading 135°, target turns to 45°). Vary the turn time from 0.02 h to 0.25 h in steps.

For each turn time, record whether the re-intercept system (fast update interval) still captures the target, and what the TOF is.

Plot turn time vs. TOF. Identify the window during which the turn is most disruptive. Answer: is there an optimal moment for the target to turn?

In [ ]:
# Exercise B

s_pos = [-98.49, 33.91]
t_pos = [-101.0, 36.5]
t_spd = 300
s_spd = 650

turn_times = [round(t * 0.01, 3) for t in range(2, 28)]  # 0.02 to 0.27 hours

tofs       = []
captured_flags = []

for turn_t in turn_times:
    sched = [(0.0, 135), (turn_t, 45)]
    r = simulate_maneuvering_target(
        s_pos, t_pos, sched, t_spd, s_spd,
        update_interval=0.005, dt=0.005, max_steps=8000
    )
    tofs.append(r["time_elapsed"] * 60 if r["captured"] else None)
    captured_flags.append(r["captured"])

valid_turn  = [t * 60 for t, tof in zip(turn_times, tofs) if tof is not None]
valid_tofs  = [tof for tof in tofs if tof is not None]
miss_turn   = [t * 60 for t, tof in zip(turn_times, tofs) if tof is None]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(valid_turn, valid_tofs, "o-", color="#2a9d8f", markersize=5, label="Captured")
if miss_turn:
    ax.scatter(miss_turn, [max(valid_tofs or [0]) * 1.1] * len(miss_turn),
               marker="x", color="#e63946", s=60, zorder=5, label="Not captured")
ax.set_xlabel("Turn time (minutes into flight)")
ax.set_ylabel("Pursuer TOF (minutes)")
ax.set_title("Effect of target turn timing on intercept TOF")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Identify most disruptive window
if valid_tofs:
    worst_idx = valid_tofs.index(max(valid_tofs))
    print(f"Most disruptive turn: at {valid_turn[worst_idx]:.1f} min  →  TOF = {valid_tofs[worst_idx]:.1f} min")

## Exercise C — Build a Defense System

Using the multi-threat prioritization code from Section 3, build a simple automated defense system.

**Rules:**
- You have one shooter that can engage one target at a time
- After each engagement (intercept computed), assume the shot is "in the air" for its TOF before the next engagement begins
- Prioritize by **earliest time to impact** (TTI)
- If a threat has no TTI (will miss the asset), skip it

Write a function `engagement_sequence(threats, shooter_pos, shooter_speed, defended_point)` that:
1. Ranks threats by TTI
2. Fires at each in order, tracking accumulated time
3. Prints the engagement order with fire time, TOF, and whether each threat was intercepted before it reached the asset

Test it on the 4-threat scenario from Section 3 plus one additional threat of your choice.

In [ ]:
# Exercise C — your code here

def engagement_sequence(threats, shooter_pos, shooter_speed, defended_point):
    """
    Simple sequential engagement planner.
    Prioritizes by TTI; skips threats that won't reach the asset.
    """
    # Compute TTI and intercept TOF for all threats
    annotated = []
    for thr in threats:
        tti = time_to_impact(thr["pos"], thr["hdg"], thr["spd"], defended_point)
        tof = find_intercept_time(shooter_pos, thr["pos"], thr["hdg"],
                                  thr["spd"], shooter_speed, t_max=15.0)
        if tti is not None:
            annotated.append({**thr, "tti_h": tti, "tof_h": tof})

    # Sort by TTI
    annotated.sort(key=lambda x: x["tti_h"])

    print(f"{'#':<3}  {'ID':<5}  {'Fire at (min)':>13}  {'TOF (min)':>10}  "
          f"{'Impact at (min)':>16}  {'Intercepted?':>13}")
    print("-" * 72)

    clock = 0.0   # hours elapsed

    for i, thr in enumerate(annotated):
        fire_time_min  = clock * 60
        tof_h          = thr["tof_h"]
        tti_min        = thr["tti_h"] * 60

        if tof_h is None:
            status = "NO SOLUTION"
            print(f"{i+1:<3}  {thr['id']:<5}  {fire_time_min:>12.1f}  "
                  f"{'—':>10}  {tti_min:>15.1f}  {status:>13}")
            continue

        arrival_min = fire_time_min + tof_h * 60
        intercepted = arrival_min < tti_min
        status = "INTERCEPTED" if intercepted else "IMPACT — too late"

        print(f"{i+1:<3}  {thr['id']:<5}  {fire_time_min:>12.1f}  "
              f"{tof_h*60:>10.1f}  {tti_min:>15.1f}  {status:>13}")

        clock += tof_h   # next engagement begins after this shot lands


# Add a 5th threat
extended_threats = threats + [
    {"id": "T5", "pos": [-98.0, 38.5], "hdg": 185, "spd": 450,
     "description": "fast, heading nearly due south — direct path to asset"},
]

engagement_sequence(extended_threats, SHOOTER_POS, SHOOTER_SPEED, DEFENDED_POINT)

---

## Check Your Understanding

A defense system uses pure pursuit to engage all incoming threats. A student proposes switching to proportional navigation to improve efficiency.

Their argument:

> *"PN is always more efficient than pure pursuit — it flies a straighter path and uses less range. We should replace pure pursuit with PN everywhere."*

**Question:** Is the student's argument completely right, partially right, or wrong? Under what conditions does PN outperform pure pursuit, and are there situations where pure pursuit might be preferred?

```python
# your answer here
```

---

## Module Complete

This notebook closes out the Intercept and Pursuit module. Across the six notebooks, you built:

| Notebook | Core contribution |
|----------|-------------------|
| 00 — Problem Setup | Scenario variables, naive miss distance, the intercept equation |
| 01 — Constant-Velocity Intercept | Binary search solver, `compute_intercept`, intercept geometry |
| 02 — Iterative Pursuit | `simulate_pursuit`, pursuit curve vs. intercept path cost |
| 03 — Visual Simulation | Interactive click-based scenario builder |
| 04 — Strategy and Limits | Escape cone, speed ratio analysis, delay budget, positioning |
| 05 — Advanced Topics | Maneuvering targets, proportional navigation, multi-threat defense |

The tools in this module — `find_intercept_time`, `simulate_pursuit`, `simulate_proportional_nav`, `engagement_sequence` — are small, composable functions built from the geographic primitives you developed in earlier modules. The geometry is real; the math is the same math that runs in actual intercept systems.